In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / "PROJECT_PLAN.md").exists(), (
    "Make sure this notebook is saved in the project's notebooks directory"
)

training_text = pd.read_parquet(
    PROJECT_ROOT / "data/processed/credit_card_training_text.parquet"
)

quality_flags = pd.read_parquet(
    PROJECT_ROOT / "data/processed/training_text_quality_flags.parquet"
)

records = training_text.merge(
    quality_flags[
        [
            "complaint_id",
            "text_group_id",
            "text_group_size",
            "has_label_conflict",
        ]
    ],
    on="complaint_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert len(records) == 33117
assert records["_merge"].eq("both").all()
assert records["has_label_conflict"].notna().all()
assert records["issue"].notna().all()

records = records.drop(columns="_merge")

print("Training records with matched quality flags:", len(records))

Training records with matched quality flags: 33117


In [2]:
# Establish a deterministic order for selecting group representatives
records = records.sort_values(
    ["received_date", "complaint_id"],
    kind="stable",
).reset_index(drop=True)

records["sample_status"] = "included"

# Exclude all training records in groups with conflicting labels
records.loc[
    records["has_label_conflict"],
    "sample_status",
] = "excluded_label_conflict"

# Deduplicate only the remaining consistent-label groups
consistent_records = records.loc[
    ~records["has_label_conflict"]
]

duplicate_mask = consistent_records["text_group_id"].duplicated(
    keep="first"
)

duplicate_indices = consistent_records.index[duplicate_mask]

records.loc[
    duplicate_indices,
    "sample_status",
] = "excluded_exact_duplicate"

baseline_training = records.loc[
    records["sample_status"].eq("included")
].copy()

sample_flow = (
    records["sample_status"]
    .value_counts()
    .reindex(
        [
            "excluded_label_conflict",
            "excluded_exact_duplicate",
            "included",
        ],
        fill_value=0,
    )
    .rename_axis("sample_status")
    .reset_index(name="rows")
)

display(sample_flow)

assert sample_flow["rows"].sum() == len(records)
assert baseline_training["complaint_id"].is_unique
assert baseline_training["text_group_id"].is_unique
assert not baseline_training["has_label_conflict"].any()

,sample_status,rows
0,excluded_label_conflict,992
1,excluded_exact_duplicate,1945
2,included,30180


In [3]:
class_summary = pd.crosstab(
    records["issue"],
    records["sample_status"],
).reindex(
    columns=[
        "included",
        "excluded_label_conflict",
        "excluded_exact_duplicate",
    ],
    fill_value=0,
)

class_summary["original_text_rows"] = class_summary.sum(axis=1)

class_summary["retention_pct"] = (
    100.0
    * class_summary["included"]
    / class_summary["original_text_rows"]
).round(2)

class_summary = class_summary[
    [
        "original_text_rows",
        "excluded_label_conflict",
        "excluded_exact_duplicate",
        "included",
        "retention_pct",
    ]
].sort_values("included", ascending=False)

print(class_summary.to_string())

print("\nOriginal issue categories:", records["issue"].nunique())
print("Retained issue categories:", baseline_training["issue"].nunique())

empty_classes = class_summary.index[
    class_summary["included"].eq(0)
].tolist()

print("Categories with no retained records:", empty_classes)

sample_status                                                    original_text_rows  excluded_label_conflict  excluded_exact_duplicate  included  retention_pct
issue                                                                                                                                                          
Problem with a purchase shown on your statement                                7791                        3                        12      7776          99.81
Fees or interest                                                               3987                        6                        12      3969          99.55
Other features, terms, or problems                                             3867                        4                         5      3858          99.77
Getting a credit card                                                          4042                       57                       288      3697          91.46
Closing your account                    

In [4]:
PROCESSED_DIR = PROJECT_ROOT / "data/processed"

BASELINE_PATH = PROCESSED_DIR / "baseline_training.parquet"
MEMBERSHIP_PATH = PROCESSED_DIR / "baseline_sample_membership.parquet"

assert len(baseline_training) == 30180
assert set(baseline_training["issue"]) == set(records["issue"])

baseline_training.to_parquet(
    BASELINE_PATH,
    index=False,
    compression="zstd",
)

# Preserve the inclusion or exclusion decision for every input record
records[
    [
        "complaint_id",
        "received_date",
        "issue",
        "text_group_id",
        "sample_status",
    ]
].to_parquet(
    MEMBERSHIP_PATH,
    index=False,
    compression="zstd",
)

saved_ids = pd.read_parquet(
    BASELINE_PATH,
    columns=["complaint_id"],
)

assert saved_ids["complaint_id"].tolist() == (
    baseline_training["complaint_id"].tolist()
)

sample_flow.to_csv(
    PROJECT_ROOT / "reports/baseline_sample_flow.csv",
    index=False,
)

class_summary.to_csv(
    PROJECT_ROOT / "reports/baseline_class_summary.csv",
    index_label="issue",
)

print("Baseline training rows saved and verified:", len(saved_ids))
print("Sample membership records saved:", len(records))

Baseline training rows saved and verified: 30180
Sample membership records saved: 33117


In [5]:
import hashlib
import json

def describe_file(path):
    """Record a project-relative path, size, and SHA-256 fingerprint."""
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return {
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256.hexdigest(),
    }

sample_config = {
    "policy_version": 1,
    "training_period": {
        "start": "2024-04-01",
        "end": "2025-02-28",
    },
    "model_input": "narrative",
    "target": "issue",
    "exclude_conflicting_training_text_groups": True,
    "exact_duplicate_selection": {
        "sort_by": ["received_date", "complaint_id"],
        "keep": "first",
    },
    "automatic_near_duplicate_removal": False,
    "minimum_text_length_filter": None,
    "counts": dict(
        zip(sample_flow["sample_status"], sample_flow["rows"].astype(int))
    ),
    "labels": sorted(baseline_training["issue"].unique().tolist()),
    "inputs": [
        describe_file(
            PROCESSED_DIR / "credit_card_training_text.parquet"
        ),
        describe_file(
            PROCESSED_DIR / "training_text_quality_flags.parquet"
        ),
    ],
    "outputs": [
        describe_file(BASELINE_PATH),
        describe_file(MEMBERSHIP_PATH),
    ],
    "sensitivity_comparison_source": (
        "data/processed/credit_card_training_text.parquet"
    ),
}

(PROJECT_ROOT / "configs/baseline_training_sample.json").write_text(
    json.dumps(sample_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Sample configuration and file fingerprints saved.")

Sample configuration and file fingerprints saved.


In [6]:
sample_report = """# Baseline Training Sample

## Construction

Training receipt dates cover April 2024 through February 2025.

- Input labeled text records: 33,117
- Excluded conflicting-label records: 992
- Excluded additional consistent-label exact duplicates: 1,945
- Retained baseline records: 30,180
- Retained issue categories: 15

For consistent-label exact duplicate groups, the earliest record is retained.
Same-date ties are resolved by complaint ID.
Near-duplicate candidates and substantive short texts are retained.

## Distribution Effects

Filtering affects categories unevenly.
The company-investigation category retains 41.95% of its text records.
The incorrect-credit-report-information category retains 68.06%.
The smallest retained category contains 18 records.

Filtering is not assumed to improve predictive performance.
Compare this sample with all labeled training records on the same validation set.

## Artifacts and Scope

The baseline sample and row-level selection decisions remain local.
Configuration, fingerprints, and aggregate reports are version controlled.

Model input is narrative; target is issue.
Original complaint events remain available for business analysis.
No classifier has been fitted or evaluated in this notebook.
"""

(PROJECT_ROOT / "reports/baseline_training_sample.md").write_text(
    sample_report,
    encoding="utf-8",
)

policy_path = PROJECT_ROOT / "docs/training_sample_policy.md"
policy = policy_path.read_text(encoding="utf-8")

policy = policy.replace(
    "## Planned Baseline Training Sample",
    "## Baseline Training Sample",
).replace(
    "This is the planned policy.\nNo baseline sample has been constructed yet.",
    "The baseline sample has been constructed with 30,180 records across "
    "15 categories.\nNo classifier has been fitted or evaluated yet.",
)

policy_path.write_text(policy, encoding="utf-8")

print("Sample report and policy status updated.")

Sample report and policy status updated.
